# Falcon-H1-0.5B route-viability screen (inference only, nothing is trained)

**Run with Runtime → Run all. Nothing to set.** The only interaction is Colab's Drive prompt.
If Colab disconnects, reconnect and run all again: each finished task is saved and skipped.

## The question

The paper predicts that post-convergence steering needs two things at once:

1. the alternative route is **independently viable** on the task, and
2. the free model is **causally dominated by the other route**, so there is something to move.

The steering pilot failed on key–value recall because condition 1 did not hold: Falcon-H1's SSM
branch could not do the binding on its own. This notebook screens several memory tasks in the
pretrained model and reports, per task, whether both conditions hold — before any steering is
attempted. Whatever it finds is usable: a task meeting both conditions is a prospectively
identified steering candidate, and the tasks that fail are negative controls for the same
prediction.

## Tasks

| task | what it asks | route expected to favour it |
|---|---|---|
| `kv_recall` | value bound to a queried key | attention (measured negative) |
| `delayed_recall` | one codeword after N tokens of filler | either |
| `state_tracking` | a variable's value after several updates | recurrence |
| `recency` | the most recently mentioned item | recurrence |
| `induction` | continue a repeated sequence | attention (expected negative) |
| `passage_fact` | a fact from a natural WikiText paragraph | either |

Delayed recall runs at several distances, so viability can be read against distance.

## Measured per task (fixed prompts, no training)

- **Full, attention-only and SSM-only accuracy.** Route-only means the *other* branch's output is
  zeroed in every layer.
- **Viability** = (route-only − chance) / (full − chance), the fraction of the model's
  above-chance performance that one branch keeps alone. Viable at ≥ `VIABLE_FRAC` (0.8).
- **D_attn, D_ssm** on the intact model, by resampling each branch from another prompt.
- **Donor rate:** how often attention resampling makes the model give the other prompt's answer.
  On key–value recall this was 1.00, the sharpest sign that attention carries the answer.
- **Bag rate** and accuracy restricted to the prompt's own candidates, so guessing cannot be
  mistaken for recall — the error that made the steering pilot look successful at first.

## Verdict per task

- **STEERING CANDIDATE** — SSM viable and attention dominant: the condition the paper predicts is
  needed. Worth a steering run.
- **BOTH ROUTES ALREADY SHARED** — SSM viable but attention not dominant: nothing to move.
- **NO ALTERNATIVE ROUTE** — SSM not viable: the boundary predicts no handover (the key–value
  case).
- **TASK NOT SOLVED** — the full model is below `SOLVED_ACC`, so the task tells us nothing.

Send back `SCREEN_SUMMARY.txt`.


In [ ]:
import hashlib, json, math, os, pathlib, random, subprocess, sys, time
from contextlib import contextmanager, nullcontext

PRESET = "full"          # leave as is ("smoke" = Falcon-H1-Tiny-90M, fewer prompts)
USE_DRIVE, DRIVE_DIR = True, "/content/drive/MyDrive/falcon_h1_screen"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = DRIVE_DIR
else:
    WORKDIR = os.environ.get("FALCON_SCREEN_WORKDIR", str(pathlib.Path.cwd() / "falcon_h1_screen"))

from importlib.metadata import version, PackageNotFoundError


def _ver(pkg):
    try:
        return tuple(int("".join(ch for ch in part if ch.isdigit()) or 0) for part in version(pkg).split(".")[:2])
    except PackageNotFoundError:
        return (0, 0)


if _ver("transformers") < (4, 53):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "accelerate"], check=True)
    raise SystemExit("transformers was upgraded. Runtime -> Restart session, then Run all again.")
if _ver("datasets") == (0, 0):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib as mpl
import matplotlib.pyplot as plt
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer


def P(full, smoke):
    return full if PRESET == "full" else smoke


MODEL = P("tiiuae/Falcon-H1-0.5B-Base", "tiiuae/Falcon-H1-Tiny-90M-Instruct")
N_PROMPTS = P(128, 16)
MAX_BATCH = 16
SEED, FORMAT_VERSION, D_FLOOR = 4242, 1, 0.10
VIABLE_FRAC, SOLVED_ACC, DOMINANT_D, SHARED_D = 0.80, 0.60, 0.70, 0.50
DELAY_DISTANCES = P([16, 64, 256], [16, 64])
KV_LOADS = P([2, 8], [2])
N_UPDATES, N_RECENCY_ITEMS, INDUCTION_LEN = 4, 6, 8

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IS_CUDA = DEVICE.type == "cuda"
GPU_NAME = torch.cuda.get_device_name(0) if IS_CUDA else "cpu"
DTYPE = (torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float32) if IS_CUDA else torch.float32

TAG = f"{MODEL.split('/')[-1]}_{PRESET}"
WORK = pathlib.Path(WORKDIR).expanduser()
RESDIR, FIGDIR, TABDIR = (WORK / f"{n}_{TAG}" for n in ("results", "figures", "tables"))
for _d in (RESDIR, FIGDIR, TABDIR):
    _d.mkdir(parents=True, exist_ok=True)
T_START = time.time()
print(f"{GPU_NAME} | {str(DTYPE).split('.')[-1]} | torch {torch.__version__} | transformers {transformers.__version__}")
print(f"{MODEL} | {N_PROMPTS} prompts per condition | results in {WORK}")


In [ ]:
def keyof(spec):
    return hashlib.md5(json.dumps(spec, sort_keys=True, default=str).encode()).hexdigest()[:16]


class Records:
    def __init__(self, path):
        self.path, self.recs = path, {}
        if path.exists():
            for line in path.read_text().splitlines():
                try:
                    r = json.loads(line); self.recs[r["_key"]] = r
                except (json.JSONDecodeError, KeyError):
                    pass

    def get(self, spec):
        return self.recs.get(keyof(spec))

    def put(self, spec, payload):
        r = {"_key": keyof(spec), "_spec": spec, "_t": time.time(), **payload}
        self.recs[r["_key"]] = r
        with open(self.path, "a") as f:
            f.write(json.dumps(r, default=float) + "\n"); f.flush(); os.fsync(f.fileno())
        return r


REC = Records(RESDIR / "records.jsonl")
BASE_SPEC = {"model": MODEL, "seed": SEED, "format": FORMAT_VERSION, "n": N_PROMPTS}
print(f"{len(REC.recs)} finished record(s) on disk")


In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=DTYPE).to(device=DEVICE, dtype=DTYPE).eval()
for p in model.parameters():
    p.requires_grad_(False)


def find_layers(m):
    n = m.config.num_hidden_layers
    lists = [(name, mod) for name, mod in m.named_modules() if isinstance(mod, torch.nn.ModuleList) and len(mod) == n]
    assert lists, f"no ModuleList of length {n}"
    return lists[0]


def find_branch(layer, preferred, patterns):
    for attr in preferred:
        mod = getattr(layer, attr, None)
        if isinstance(mod, torch.nn.Module):
            return attr, mod
    hits = [(n, c) for n, c in layer.named_children() if any(p in type(c).__name__.lower() for p in patterns)]
    assert len(hits) == 1, f"cannot identify {patterns}: {[(n, type(c).__name__) for n, c in layer.named_children()]}"
    return hits[0]


LAYER_PATH, LAYERS = find_layers(model)
ATTN = [find_branch(l, ("self_attn",), ("attention",)) for l in LAYERS]
SSM = [find_branch(l, ("mamba",), ("mamba", "mixer", "ssm")) for l in LAYERS]
N_LAYERS = len(LAYERS)
CFG = model.config
ATTN_CFG = {k: getattr(CFG, k, None) for k in
            ("max_position_embeddings", "sliding_window", "use_sliding_window", "layer_types",
             "num_attention_heads", "num_key_value_heads", "rope_theta")}
ATTN_CFG = {k: (v if not isinstance(v, list) else sorted(set(v))) for k, v in ATTN_CFG.items() if v is not None}
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters | {N_LAYERS} layers | "
      f"attention .{ATTN[0][0]} | SSM .{SSM[0][0]}")
print(f"attention config: {ATTN_CFG}")

ACTIVE = {}


def _hook(route, idx):
    def fn(module, inputs, output):
        spec = ACTIVE.get(route)
        if spec is None:
            return None
        t = output[0] if isinstance(output, tuple) else output
        kind = spec["kind"]
        if kind == "record":
            spec["store"].append(t.detach().clone())
            return None
        if kind == "zero":
            new = torch.zeros_like(t)
        elif kind == "cache":
            new = spec["store"][idx][spec["perm"]]
        else:
            raise ValueError(kind)
        return (new,) + tuple(output[1:]) if isinstance(output, tuple) else new
    return fn


for _h in globals().get("HOOK_HANDLES", []):
    _h.remove()
HOOK_HANDLES = ([mod.register_forward_hook(_hook("attn", i)) for i, (_, mod) in enumerate(ATTN)]
                + [mod.register_forward_hook(_hook("ssm", i)) for i, (_, mod) in enumerate(SSM)])


@contextmanager
def intervene(**routes):
    ACTIVE.clear()
    ACTIVE.update({k: v for k, v in routes.items() if v is not None})
    try:
        yield
    finally:
        ACTIVE.clear()


@torch.no_grad()
def last_logits(x):
    for kw in ({"logits_to_keep": 1}, {"num_logits_to_keep": 1}, {}):
        try:
            return model(input_ids=x, use_cache=False, **kw).logits[:, -1, :].float()
        except TypeError:
            continue
    raise RuntimeError("model forward failed")


def sync():
    if IS_CUDA:
        torch.cuda.synchronize()


def slices(n, B):
    B = max(2, B)
    sl = [(s, min(s + B, n)) for s in range(0, n, B)]
    if len(sl) > 1 and sl[-1][1] - sl[-1][0] == 1:
        sl[-2] = (sl[-2][0], n); sl.pop()
    return sl


def with_oom_retry(fn, B, label):
    while True:
        try:
            return fn(B)
        except torch.cuda.OutOfMemoryError:
            ACTIVE.clear()
            if IS_CUDA:
                torch.cuda.empty_cache()
            if B <= 2:
                raise
            B //= 2
            print(f"    out of memory in {label}; retrying with batch {B}", flush=True)


In [ ]:
# Each task builds prompts of equal length within a condition (no padding) and returns, per prompt:
# the token ids, the answer token, and the candidate set the answer is drawn from. Accuracy is
# scored over the candidate set, so chance is 1/len(candidates) and "emitting a plausible token"
# cannot be mistaken for solving the task.
NOUNS = """apple river table window garden doctor pencil island forest bottle mirror candle bridge castle rocket
tiger lemon piano jacket helmet market pocket village teacher engine planet carpet button ticket basket hammer
ladder wallet bucket camera coffee cookie dragon farmer finger flower guitar hunter kitten letter monkey needle
orange parrot pillow puzzle rabbit saddle sister spider statue sugar thunder tomato turtle wagon wizard anchor
arrow badge banner barrel beach beard bell belt bench blanket boat bone book boot bread brick brush cabin cable
cake canal cannon card cart chain chair cheese cherry chest chicken church cloud coat coin comet cotton crown cup
curtain desk diamond dinner dog door drum eagle egg elbow fence field flag flute fork fox frog glass glove goat
grape hat honey horse house iron jar jewel kettle key kite knife lake lamp leaf lion lock map mask milk moon
mouse nail nest net oven owl paint palace paper pearl pepper pig pipe plate pond pot queen rain ring road robot
roof rope rose salt scarf ship shirt shoe shovel sink skirt snake soap sock spoon star stone storm sword tent
tower train tree truck tunnel violin wall whale wheel wolf yard""".split()
ADJS = """red blue green yellow purple brown black white pink gray golden silent happy angry brave calm clever
cold dark eager early empty fair fancy fast fierce gentle giant grand great heavy hollow humble icy jolly kind
large late lazy little lonely loud lucky mild modern narrow neat noble odd old plain polite proud quick quiet
rapid rare rich rough round royal rusty sacred safe sharp shiny short shy simple sleepy slow small smooth soft
solid sour spicy steep sticky stormy strange strict strong sunny sweet swift tall tame thick thin tidy tiny tough
ugly vast warm weak wet wide wild wise young bitter bold bright broad busy cheap crisp cruel cute damp deep dry
dusty faint false fresh frozen full funny grumpy hairy harsh hot huge juicy keen lean lively loyal mad merry
messy misty muddy nasty naughty nervous nice pale pure rigid ripe rude salty scary shallow silly skinny spotty
stale stiff sturdy tender tense tricky vivid weird wooden zany""".split()
NUMBERS = [str(i) for i in range(10, 100)]


def single_token(words, exclude=()):
    ids, kept = [], []
    for w in words:
        e = tok.encode(" " + w, add_special_tokens=False)
        if len(e) == 1 and e[0] not in ids and e[0] not in exclude:
            ids.append(e[0]); kept.append(w)
    return ids, kept


NOUN_IDS, NOUN_W = single_token(NOUNS)
ADJ_IDS, ADJ_W = single_token(ADJS, exclude=set(NOUN_IDS))
NUM_IDS, NUM_W = single_token(NUMBERS)
MIN_POOL = max(16, max(KV_LOADS), N_RECENCY_ITEMS, INDUCTION_LEN + 1)
assert len(NOUN_IDS) >= MIN_POOL and len(ADJ_IDS) >= MIN_POOL, (
    f"word pools too small for this tokenizer: {len(NOUN_IDS)} nouns, {len(ADJ_IDS)} adjectives, "
    f"need {MIN_POOL} of each")
# Many tokenizers split multi-digit numbers into separate digit tokens, which would make the answer
# span several tokens. When that happens the state-tracking task uses words instead of numbers.
if len(NUM_IDS) >= MIN_POOL:
    STATE_IDS, STATE_NOUN, STATE_KIND = NUM_IDS, "counter", "numbers"
else:
    STATE_IDS, STATE_NOUN, STATE_KIND = ADJ_IDS, "flag", "words"
print(f"single-token pools: {len(NOUN_IDS)} nouns, {len(ADJ_IDS)} adjectives, {len(NUM_IDS)} numbers "
      f"-> state tracking uses {STATE_KIND} (the {STATE_NOUN})")


def piece(s):
    return tok.encode(s, add_special_tokens=False)


BOS = [tok.bos_token_id] if tok.bos_token_id is not None else []
_FILLER = piece(" The weather stayed mild and the town went about its usual business without any hurry at all.")


def filler(n_tokens, rng):
    out = []
    while len(out) < n_tokens:
        start = int(rng.integers(len(_FILLER)))
        out += _FILLER[start:]
    return out[:n_tokens]


def task_kv_recall(rng, load):
    ids = BOS + piece("Memorize the following pairs.\n")
    ks = rng.choice(len(NOUN_IDS), load, replace=False)
    vs = rng.choice(len(ADJ_IDS), load, replace=False)
    for k, v in zip(ks, vs):
        ids += piece("-") + [NOUN_IDS[k]] + piece(":") + [ADJ_IDS[v]] + piece("\n")
    q = int(rng.integers(load))
    ids += piece("Now recall one of them.\n") + piece("-") + [NOUN_IDS[ks[q]]] + piece(":")
    return ids, ADJ_IDS[vs[q]], [ADJ_IDS[v] for v in vs]


def task_delayed_recall(rng, dist):
    cands = rng.choice(len(ADJ_IDS), 16, replace=False)
    t = int(rng.integers(16))
    ids = (BOS + piece("Remember this codeword:") + [ADJ_IDS[cands[t]]] + piece(".\n")
           + filler(dist, rng) + piece("\nQuestion: the codeword was:"))
    return ids, ADJ_IDS[cands[t]], [ADJ_IDS[c] for c in cands]


def task_state_tracking(rng, n_updates):
    # the answer is the LAST value assigned; earlier values are the distractor candidates
    cands = rng.choice(len(STATE_IDS), 16, replace=False)
    vals = cands[:n_updates]
    ids = BOS + piece(f"The {STATE_NOUN} changes over time.\n")
    for v in vals:
        ids += piece(f"Set the {STATE_NOUN} to") + [STATE_IDS[v]] + piece(".\n")
    ids += piece(f"The {STATE_NOUN} is now:")
    return ids, STATE_IDS[vals[-1]], [STATE_IDS[c] for c in cands]


def task_recency(rng, n_items):
    items = rng.choice(len(NOUN_IDS), n_items, replace=False)
    ids = BOS + piece("A list of things seen, in order.\n")
    for it in items:
        ids += piece("- I saw a") + [NOUN_IDS[it]] + piece(".\n")
    ids += piece("The last thing I saw was a")
    return ids, NOUN_IDS[items[-1]], [NOUN_IDS[i] for i in items]


def task_induction(rng, length):
    seq = rng.choice(len(NOUN_IDS), length, replace=False)
    cut = int(rng.integers(1, length))
    ids = BOS + piece("Repeat the sequence.\n")
    for s in seq:
        ids += [NOUN_IDS[s]]
    ids += piece("\n") + [NOUN_IDS[s] for s in seq[:cut]]
    return ids, NOUN_IDS[seq[cut]], [NOUN_IDS[s] for s in seq]


_WIKI_PARAS = None


def task_passage_fact(rng, dist):
    # a codeword inserted into a natural WikiText paragraph, queried afterwards
    global _WIKI_PARAS
    if _WIKI_PARAS is None:
        from datasets import load_dataset
        try:
            ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="validation")
        except Exception:
            ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="validation")
        paras = [p for p in ds["text"] if len(p.split()) > 120 and not p.strip().startswith("=")]
        _WIKI_PARAS = [piece(p)[:512] for p in paras[:400]]
    cands = rng.choice(len(ADJ_IDS), 16, replace=False)
    t = int(rng.integers(16))
    para = _WIKI_PARAS[int(rng.integers(len(_WIKI_PARAS)))]
    body = (para * (dist // max(len(para), 1) + 1))[:dist]
    ids = (BOS + piece("Remember this codeword:") + [ADJ_IDS[cands[t]]] + piece(".\n") + body
           + piece("\nQuestion: the codeword was:"))
    return ids, ADJ_IDS[cands[t]], [ADJ_IDS[c] for c in cands]


TASKS = ([("kv_recall", lambda r, p=p: task_kv_recall(r, p), f"{p} pairs") for p in KV_LOADS]
         + [("delayed_recall", lambda r, d=d: task_delayed_recall(r, d), f"{d} tokens") for d in DELAY_DISTANCES]
         + [("state_tracking", lambda r: task_state_tracking(r, N_UPDATES), f"{N_UPDATES} updates"),
            ("recency", lambda r: task_recency(r, N_RECENCY_ITEMS), f"{N_RECENCY_ITEMS} items"),
            ("induction", lambda r: task_induction(r, INDUCTION_LEN), f"{INDUCTION_LEN} items")]
         + [("passage_fact", lambda r, d=d: task_passage_fact(r, d), f"{d} tokens")
            for d in DELAY_DISTANCES if d >= 64])


def build(task_fn, n, seed):
    rng = np.random.default_rng(seed)
    rows = [task_fn(rng) for _ in range(n)]
    L = max(len(r[0]) for r in rows)
    rows = [r for r in rows if len(r[0]) == L] or rows        # keep one length: no padding
    nc = min(len(r[2]) for r in rows)
    X = torch.tensor([r[0] for r in rows])
    Y = torch.tensor([r[1] for r in rows])
    C = torch.tensor([r[2][:nc] for r in rows])
    return X, Y, C


for name, fn, cond in TASKS:
    X, Y, C = build(fn, 4, SEED)
    print(f"\n--- {name} ({cond}): {X.shape[1]} tokens, {C.shape[1]} candidates, chance {1 / C.shape[1]:.3f}")
    txt = tok.decode(X[0])
    print(("   " + (txt if len(txt) < 420 else txt[:210] + "  [...]  " + txt[-210:])).replace("\n", " / ")
          + f"   ->{tok.decode([int(Y[0])])}")


In [ ]:
# Self-test on one task: hooks fire per layer, identity resampling is exact, each intervention bites.
_X, _Y, _C = build(TASKS[0][1], 4, SEED)
_x = _X.to(DEVICE)
_rec = {"attn": [], "ssm": []}
with intervene(attn={"kind": "record", "store": _rec["attn"]}, ssm={"kind": "record", "store": _rec["ssm"]}):
    _clean = last_logits(_x)
assert len(_rec["attn"]) == N_LAYERS == len(_rec["ssm"]), "hooks did not fire once per layer"
_tol = (1e-4 if DTYPE == torch.float32 else 5e-3) * max(1.0, float(_clean.abs().max()))
_id = torch.arange(len(_x), device=DEVICE)
with intervene(attn={"kind": "cache", "store": _rec["attn"], "perm": _id},
               ssm={"kind": "cache", "store": _rec["ssm"], "perm": _id}):
    assert float((last_logits(_x) - _clean).abs().max()) <= _tol, "identity resample changed the output"
for _r in ("attn", "ssm"):
    with intervene(**{_r: {"kind": "zero"}}):
        assert float((last_logits(_x) - _clean).abs().max()) > 10 * _tol, f"deleting {_r} had no effect"
with intervene(attn={"kind": "cache", "store": _rec["attn"], "perm": torch.roll(_id, 1)}):
    assert float((last_logits(_x) - _clean).abs().max()) > 10 * _tol, "attention resampling had no effect"
del _rec
if IS_CUDA:
    torch.cuda.empty_cache()
print(f"self-test passed: {N_LAYERS} hooks per route, identity resample exact, deletion and resampling both bite")


In [ ]:
CONDS = ["clean", "attn_res", "ssm_res", "attn_only", "ssm_only"]
IV = {"clean": {}, "attn_only": {"ssm": {"kind": "zero"}}, "ssm_only": {"attn": {"kind": "zero"}}}


def score(lg, y, cand, y_donor):
    pred = cand.gather(1, torch.gather(lg, 1, cand).argmax(-1, keepdim=True)).squeeze(1)
    top = lg.argmax(-1)
    return {"acc": float((pred == y).sum()),            # forced choice among this prompt's candidates
            "top1": float((top == y).sum()),            # unconstrained next token
            "in_cand": float((top[:, None] == cand).any(1).sum()),
            "donor": float((pred == y_donor).sum())}


def measure(task_fn, B):
    X, Y, C = build(task_fn, N_PROMPTS, SEED)
    n, chance = len(X), 1.0 / C.shape[1]
    tot = {c: dict.fromkeys(["acc", "top1", "in_cand", "donor"], 0.0) for c in CONDS}
    sync(); t0 = time.time()
    for i, j in slices(n, B):
        x, y, c = X[i:j].to(DEVICE), Y[i:j].to(DEVICE), C[i:j].to(DEVICE)
        perm = torch.roll(torch.arange(x.shape[0], device=DEVICE), 1)
        y_d = y[perm]
        rec = {"attn": [], "ssm": []}
        with intervene(attn={"kind": "record", "store": rec["attn"]}, ssm={"kind": "record", "store": rec["ssm"]}):
            runs = {"clean": last_logits(x)}
        with intervene(attn={"kind": "cache", "store": rec["attn"], "perm": perm}):
            runs["attn_res"] = last_logits(x)
        with intervene(ssm={"kind": "cache", "store": rec["ssm"], "perm": perm}):
            runs["ssm_res"] = last_logits(x)
        del rec
        for cond in ("attn_only", "ssm_only"):
            with intervene(**IV[cond]):
                runs[cond] = last_logits(x)
        for cond, lg in runs.items():
            for k, v in score(lg, y, c, y_d).items():
                tot[cond][k] += v
        if IS_CUDA:
            torch.cuda.empty_cache()
    sync()
    out = {"n": n, "tokens": int(X.shape[1]), "candidates": int(C.shape[1]), "chance": chance,
           "seconds": time.time() - t0, "batch": max(2, B)}
    for cond in CONDS:
        for k, v in tot[cond].items():
            out[f"{cond}_{k}"] = v / n
    head = max(out["clean_acc"] - chance, 1e-9)
    out["viab_attn"] = (out["attn_only_acc"] - chance) / head
    out["viab_ssm"] = (out["ssm_only_acc"] - chance) / head
    for r in ("attn", "ssm"):
        out[f"D_{r}"] = (float("nan") if out["clean_acc"] - chance < D_FLOOR
                         else float(np.clip((out["clean_acc"] - out[f"{r}_res_acc"]) / head, 0.0, 1.0)))
    return out


ROWS = []
print(f"{'task':<16}{'cond':>12}{'tok':>6}{'chance':>8}{'full':>7}{'attnOnly':>9}{'ssmOnly':>8}"
      f"{'viabA':>7}{'viabS':>7}{'D_a':>6}{'D_s':>6}{'donor':>7}{'sec':>6}")
for name, fn, cond in TASKS:
    spec = {**BASE_SPEC, "task": name, "cond": cond}
    r = REC.get(spec)
    if r is None:
        r = REC.put(spec, with_oom_retry(lambda B, fn=fn: measure(fn, B), MAX_BATCH, f"{name} {cond}"))
    ROWS.append({"task": name, "cond": cond, **{k: v for k, v in r.items() if not k.startswith("_")}})
    print(f"{name:<16}{cond:>12}{r['tokens']:>6}{r['chance']:>8.3f}{r['clean_acc']:>7.3f}{r['attn_only_acc']:>9.3f}"
          f"{r['ssm_only_acc']:>8.3f}{r['viab_attn']:>7.2f}{r['viab_ssm']:>7.2f}{r['D_attn']:>6.2f}"
          f"{r['D_ssm']:>6.2f}{r['attn_res_donor']:>7.2f}{r['seconds']:>6.0f}", flush=True)


In [ ]:
S = pd.DataFrame(ROWS)
S["solved"] = S.clean_acc >= SOLVED_ACC
S["ssm_viable"] = S.solved & (S.viab_ssm >= VIABLE_FRAC)
S["attn_viable"] = S.solved & (S.viab_attn >= VIABLE_FRAC)
S["attn_dominant"] = S.solved & (S.D_attn >= DOMINANT_D) & ((S.D_ssm <= SHARED_D) | (S.D_attn - S.D_ssm >= 0.2))


def verdict(r):
    if not r["solved"]:
        return "TASK NOT SOLVED"
    if not r["ssm_viable"]:
        return "NO ALTERNATIVE ROUTE"
    return "STEERING CANDIDATE" if r["attn_dominant"] else "BOTH ROUTES ALREADY SHARED"


S["verdict"] = S.apply(verdict, axis=1)
S.to_csv(TABDIR / "screen_by_task.csv", index=False)

mpl.rcParams.update({"figure.dpi": 130, "savefig.dpi": 300, "savefig.bbox": "tight", "font.size": 8,
                     "axes.labelsize": 8, "axes.titlesize": 8.5, "legend.fontsize": 7, "xtick.labelsize": 7,
                     "ytick.labelsize": 7, "axes.spines.top": False, "axes.spines.right": False})
VCOL = {"STEERING CANDIDATE": "#3a9d5d", "BOTH ROUTES ALREADY SHARED": "#1b6ca8",
        "NO ALTERNATIVE ROUTE": "#e07a1f", "TASK NOT SOLVED": "#999999"}
fig, ax = plt.subplots(1, 2, figsize=(7.0, 3.0))
lab = [f"{t}\n{c}" for t, c in zip(S.task, S.cond)]
y = np.arange(len(S))
ax[0].barh(y - 0.2, S.viab_attn.clip(0, 1.2), 0.38, color="#e07a1f", label="attention alone")
ax[0].barh(y + 0.2, S.viab_ssm.clip(0, 1.2), 0.38, color="#1b6ca8", label="SSM alone")
ax[0].axvline(VIABLE_FRAC, color="0.4", lw=0.8, ls="--")
ax[0].set_yticks(y); ax[0].set_yticklabels(lab); ax[0].invert_yaxis()
ax[0].set_xlabel("fraction of the model's above-chance accuracy kept alone")
ax[0].set_title("route viability"); ax[0].legend(frameon=False, loc="lower right")
for i, r in S.reset_index().iterrows():
    ax[1].scatter(r["viab_ssm"], r["D_attn"], s=26, color=VCOL[r["verdict"]], zorder=3)
    ax[1].annotate(f"{r['task']}\n{r['cond']}", (r["viab_ssm"], r["D_attn"]), fontsize=5.5,
                   xytext=(3, 3), textcoords="offset points")
ax[1].axvline(VIABLE_FRAC, color="0.4", lw=0.8, ls="--")
ax[1].axhline(DOMINANT_D, color="0.4", lw=0.8, ls="--")
ax[1].set_xlabel("SSM viability"); ax[1].set_ylabel("attention dependence $D_a$")
ax[1].set_title("steering candidates: right of and above the lines")
ax[1].set_xlim(-0.15, 1.25); ax[1].set_ylim(-0.05, 1.08)
fig.tight_layout()
for ext in ("pdf", "png"):
    fig.savefig(FIGDIR / f"falcon_route_screen.{ext}")
plt.show()

cols = ["task", "cond", "tokens", "candidates", "chance", "clean_acc", "attn_only_acc", "ssm_only_acc",
        "viab_attn", "viab_ssm", "D_attn", "D_ssm", "attn_res_donor", "ssm_res_donor", "clean_in_cand",
        "attn_only_in_cand", "ssm_only_in_cand", "verdict"]
cands = S[S.verdict == "STEERING CANDIDATE"]
lines = [f"=== FALCON-H1 ROUTE-VIABILITY SCREEN | {time.strftime('%Y-%m-%d %H:%M')} ===",
         f"model {MODEL} | {GPU_NAME} | {str(DTYPE).split('.')[-1]} | transformers {transformers.__version__} | "
         f"{N_LAYERS} layers | {N_PROMPTS} prompts per condition | {(time.time() - T_START) / 60:.0f} min",
         f"attention config: {ATTN_CFG}",
         f"thresholds: solved acc >= {SOLVED_ACC}; route viable if it keeps >= {VIABLE_FRAC} of above-chance "
         f"accuracy alone; attention dominant if D_a >= {DOMINANT_D} and (D_s <= {SHARED_D} or D_a - D_s >= 0.2)",
         "", "accuracy is a forced choice among each prompt's own candidates (chance shown per task);",
         "*_in_cand is how often the unconstrained next token is one of those candidates.", "",
         S[cols].round(3).to_string(index=False), "",
         f"steering candidates: {'none' if not len(cands) else ', '.join(f'{r.task} ({r.cond})' for r in cands.itertuples())}"]
if len(cands):
    b = cands.sort_values(["viab_ssm", "D_attn"], ascending=False).iloc[0]
    lines.append(f"best candidate: {b['task']} ({b['cond']}) -- full {b['clean_acc']:.3f}, SSM alone "
                 f"{b['ssm_only_acc']:.3f} (viability {b['viab_ssm']:.2f}), attention alone {b['attn_only_acc']:.3f}, "
                 f"D_attn {b['D_attn']:.2f}, D_ssm {b['D_ssm']:.2f}, donor {b['attn_res_donor']:.2f}")
else:
    lines.append("no task in this screen meets both conditions; the boundary predicts no handover for any of them")
SUMMARY = "\n".join(lines)
(TABDIR / "SCREEN_SUMMARY.txt").write_text(SUMMARY)
print(SUMMARY)
print(f"\nsaved {TABDIR / 'SCREEN_SUMMARY.txt'}, {TABDIR / 'screen_by_task.csv'}, {FIGDIR}")
